# TP1 - Aprendizaje automatico - KNN

## Bibliotecas a utilizar en el cuaderno

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, KFold, cross_val_score, ShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots

Creación de un dataframe a partir del archivo csv

In [ ]:
dfIris = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vSLgU6YF5djPgcJvcmXyqdIjfVefPsYlj6HUnRH15sZwsEL4GX7KPY-c3CWgM3n8vCljid-ZPocdAAl/pub?output=csv')

## Preprocesamiento

### Feature Scaling
- No se hace uso de un StandardScaler o MinMaxScaler para unificar las escalas porque en el dataset estas ya estan informadas en la misma escala (centímetros).

### Header standarization & data cleaning

- Renombraremos las columnas al español, y eliminaremos la columan Id ya que no será utilizada para el análisis

In [ ]:
#Renombrado de columnas
dfIris.rename({'SepalLengthCm':'sepalo_largo',
           'SepalWidthCm':'sepalo_ancho',
           'PetalLengthCm':'petalo_largo',
           'PetalWidthCm':'petalo_ancho',
           'Species':'especies'},
          axis=1, inplace=True) # inplace = True para que el renombrado sea sobre el mismo df

dfIris.drop('Id', axis=1, inplace=True) #axis=1 indica que es una columna


Inspección de la forma del df

In [ ]:
dfIris.shape

Visualización del segmento inicial y final de los datos

In [ ]:
dfIris

In [ ]:
sns.pairplot(dfIris, hue="especies", diag_kind="kde")

**Una particularidad que tiene Knn que la variable objetivo no puede ser categòrica, por lo tanto para poder usuarlo como clasificador se debe Codificar para que sea numèrica**


In [ ]:
# Separación de datos en dos dataframes. En "X" los atributos, en "y" las etiquetas
X = dfIris[['sepalo_ancho','petalo_largo']].values
y = dfIris['especies'].values
X.shape, y.shape

In [ ]:
le = LabelEncoder()
y = le.fit_transform(y) #transforma las etiquetas en còdigos numericos
print (y)

## Cross-validation (Con 2 parámetros)

### Seleccion de k y splits

- k: número de vecinos (n_neighbors)
- splits: cantidad de segmentos en los que se divide al dataset (n_splits)

In [ ]:
# 1. Instanciamos el modelo con n_neighbors = 12
model_knn = KNeighborsClassifier(n_neighbors=12)

# 2. Definimos la estrategia de partición / resampling method
# n_splits=5 es la cantidad de particiones, subconjuntos o pliegues
kf = KFold(n_splits=5, shuffle=True, random_state=88)

### Entrenamiento y  testeo

In [ ]:
y_pred = [None] * kf.n_splits
y_test = [None] * kf.n_splits
for i, (train_index, test_index) in enumerate(kf.split(X)):
  y_test[i] = y[test_index]
  model_knn.fit(X[train_index], y[train_index])
  y_pred[i] = model_knn.predict(X[test_index])

### Resultados del entrenamiento y contraste con valores esperados

In [ ]:
for i in np.arange(len(y_test)):
  print(f"Entrenamiento {i+1}")
  print(classification_report(y_test[i], y_pred[i]))
  print("--------")
  print(confusion_matrix(y_test[i], y_pred[i]))
  print("--------------------------------------------------------")


### Evaluación de resultados. Promedio de precisión y desviación estandar

In [ ]:
scores = cross_val_score(model_knn, X, y, cv=kf)
promedio = scores.mean()
print(f"Precisión promedio (más robusta): {promedio:.4f}")

desviacion_estandar = scores.std()
print(f"Desviación estándar: {desviacion_estandar:.4f}")

### Conclusiones parciales

Si observamos las matrices de confusión:

- Entrenamiento 2: Confundió 2 muestras de la Clase 1 con la Clase 2.

- Entrenamiento 3: Confundió 2 muestras de la Clase 1 con la Clase 2 (curiosamente el mismo error, pero visto desde la métrica de la Clase 2).

- Entrenamiento 5: Hubo un intercambio mutuo (1 de la Clase 1 pasó a la 2, y 1 de la Clase 2 pasó a la 1).

Esto indica que las clases 1 y 2 están "solapadas". Hay vecinos de una clase metidos en el territorio de la otra.

El modelo presenta una variabilidad que depende de la partición de los datos utilizada para el entrenamiento. En los folds 2, 3 y 5, la presencia de muestras en zonas de frontera (donde las clases 1 y 2 se traslapan) provoca errores de clasificación. Dado que el tamaño de la muestra por fold es pequeño (n=30), cada error individual penaliza el accuracy en un 3.3%, lo que explica las caídas de 1.0 a 0.93. La Clase 0 es la única perfectamente separable en todos los escenarios.


## Cross-validation con los 4 parámetros

Repetimos el experimento aumentando el total de parámetros de 2 a 4, es decir empleando las cuatro columnas de datos disponibles en el dataset y conservando k=12

In [ ]:
# Separo en X los atributos, en Y las etiquetas
X4 = dfIris[['sepalo_largo','sepalo_ancho','petalo_largo','petalo_ancho']].values

# 1. Instanciamos el modelo
model_knn = KNeighborsClassifier(n_neighbors=12)

# 2. Configuramos el "generador" de los experimentos de la imagen
kf4 = KFold(n_splits=5, shuffle=True, random_state=88)

# 3. Ejecutamos la validación cruzada sobre TODO el dataset (X4 e y)
scores4 = cross_val_score(model_knn, X4, y, cv=kf4)

print(f"Resultados de cada experimento: {scores4}")
print(f"Precisión promedio (más robusta): {scores4.mean():.4f}")
print(f"Desv. est.: {scores4.std():.4f}")

In [ ]:
y_pred4 = [None] * kf4.n_splits
y_test4 = [None] * kf4.n_splits
for i, (train_index, test_index) in enumerate(kf4.split(X4)):
  y_test4[i] = y[test_index]
  model_knn.fit(X4[train_index], y[train_index])
  y_pred4[i] = model_knn.predict(X4[test_index])

for i in np.arange(len(y_test4)):
  print(f"Entrenamiento {i+1}")
  print(classification_report(y_test4[i], y_pred4[i]))
  print("--------")
  print(confusion_matrix(y_test4[i], y_pred4[i]))
  print("--------------------------------------------------------")


### Conclusión parcial

Al incrementar la cantidad de dimensiones en el entrenamiento manteniendo el resto de los parámetros experimentales previos obtuvimos una mismo precisión promedio y una mayor desviación estándar que pasó de 0.0327 a 0.0389 respectivamente.

Este incremento de desviación estándar se puede evidenciar en el entrenamiento 3 dado que este manifiesta 3 errores a diferencia del caso anterior con 2 dimensiones en el que ningún entrenamiento habia superado 2 errores.

## Estudio del comportamiento de la precisión y sensibilidad en función de la cantidad de vecinos (usando registros de: 4 dimensiones)

In [ ]:
# Separamos el dataframe original en dos porciones para tener un blind fold
# El blind fold consta de 10 lineas de cada especie (b_fold)
# El dataframe que se untiliza para entrenamiento y testeo consta de 40 lineas de cada especie (t_fold)

# NOTA:
# Al explorar las series de datos de dfIris notamos que no respondian a un orden
# relativo a ninguna dimensión observada con lo cual consideramos que la distribución
# de los registros en el dataset era aleatoria y por eso tomamos los primeros 10 registros
# correspondientes a cada especie a continuación como muestra aleatoria:
b_fold = dfIris.groupby('especies').head(10)
# De no haber sido aleatoria la ortodoxia metodologia indicara utilizar en su lugar:
# b_fold = dfIris.groupby('especies').sample(n=10, random_state=42)

t_fold = dfIris.drop(b_fold.index)

In [ ]:
b_fold.info()

In [ ]:
X4 = t_fold[['sepalo_largo','sepalo_ancho','petalo_largo','petalo_ancho']].values
y = t_fold['especies'].values
y = le.fit_transform(y)


In [ ]:
desvest_output = []
accuracy_history = []
neighbors = []

for i in range(2,30) :
  # 1. Instanciamos el modelo
  model_knn = KNeighborsClassifier(n_neighbors= i)

  # 2. Configuramos el "generador" de los experimentos de la imagen
  kf = KFold(n_splits=5, shuffle=True, random_state=88)

  # 3. Ejecutamos la validación cruzada sobre TODO el dataset (X4 e y)
  scores = cross_val_score(model_knn, X4, y, cv=kf)

  desvest_output.append(scores.std())
  accuracy_history.append(scores.mean())
  neighbors.append(i)

  # imprime neighbors y desvest_output
  mean_score = scores.mean()
  std_score = scores.std()
  print(f"Vecinos (K): {i} | Accuracy: {mean_score:.4f} | Desv. Est: {std_score:.4f}")

fig, ax1 = plt.subplots(figsize=(10, 6))

# --- Configuración del Eje 1 (Precisión Promedio) ---
color_acc = 'tab:blue'
ax1.set_xlabel('Número de vecinos (K)')
ax1.set_ylabel('Precisión Promedio', color=color_acc, fontsize=12, fontweight='bold')
ax1.plot(neighbors, accuracy_history, color=color_acc, marker='s', label='Precisión')
ax1.tick_params(axis='y', labelcolor=color_acc)
# Rango: 0.92 a 1
ax1.set_ylim(0.92, 0.99)

# --- Configuración del Eje 2 (Desviación Estándar) ---
ax2 = ax1.twinx()  # Crea el segundo eje compartiendo el mismo eje X
color_std = 'tab:red'
ax2.set_ylabel('Desviación Estándar', color=color_std, fontsize=12, fontweight='bold')
ax2.plot(neighbors, desvest_output, color=color_std, marker='o', linestyle='--', label='Desv. Estándar')
ax2.tick_params(axis='y', labelcolor=color_std)
# Rango solicitado: 0.02 a 0.08
ax2.set_ylim(0.015, 0.06)

# --- Estética Final ---
plt.title('Impacto de K en la estabilidad y precisión del KNN', fontsize=14)
ax1.grid(True, linestyle='--', alpha=0.6)
fig.tight_layout() # Ajusta los márgenes para que no se corten las etiquetas
plt.show()


### Conclusión parcial:

* Se puede observar que si bien cada una de las tres
especies conserva su misma representanción relativa respecto de las demás, al realizar el entrenamiento con un universo mas reducido de datos el gráfico cambia sustancialmente y la cantidad de n_neighboors que pasan a devolver el maximo valor de acuracy corresponde a n_neighbors = 14 y 16.

* Se observa una zona amplia de meseta con n_neighbors >= 9 y n_neighbors =< 13

### Utilizamos k=14 (que corresponde a un máximo de acuracy) para entrenar un nuevo modelo para realizar predicciones sobre nuestro lote ciego para auditar los resultados

In [ ]:
# 1. Preparamos el modelo definitivo con el mejor K según tu gráfica
mejor_k = 14
modelo_k_14= KNeighborsClassifier(n_neighbors=mejor_k)

# 2. Entrenamos el modelo con TODO el t_fold (las 120 líneas)
# Usamos X4 e y que ya definiste antes
modelo_k_14.fit(X4, y)

# 3. Preparamos los datos del 'blind fold' (b_fold)
# Extraemos las características (features)
X_blind = b_fold[['sepalo_largo','sepalo_ancho','petalo_largo','petalo_ancho']].values
# Transformamos las etiquetas (species) usando el LabelEncoder que ya tienes
y_blind = le.transform(b_fold['especies'].values)

# 4. El examen final: predecimos sobre datos que el modelo NUNCA vio
predicciones_k_14 = modelo_k_14.predict(X_blind)

# 5. Calculamos la precisión real

precision_real = accuracy_score(y_blind, predicciones_k_14)

print(f"--- RESULTADO FINAL ---")
print(f"Precisión en el Lote Ciego: {precision_real * 100:.2f}%")

### Matriz de confisión sobre el lote ciego

In [ ]:
# Generamos la matriz
cm = confusion_matrix(y_blind, predicciones_k_14)

# La graficamos para que sea legible
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicción del Modelo')
plt.ylabel('Clase Real (La Verdad)')
plt.title(f'¿En qué falló el modelo con K={mejor_k}?')
plt.show()

índices de los fallos

In [ ]:
indices_fallo_14 = np.where(predicciones_k_14 != y_blind)[0]
print(f"Índices de error con K=14: {indices_fallo_14}")
# total de errores
print(f"Total de errores con K=14: {len(indices_fallo_14)}")

### Comparacion de capacidad predictiva con un K que pertenece a una meseta de la gráfica de 'Impacto de K en la estabilidad y precisión del KNN'

En este caso elejimos de la zona de meseta, encontramos al valor k=10 que se corresponde al menor entero del resultado de la raiz cuadrada de 120 que es el tamaño de nuestro dataset de entrenamiento (t_fold).

In [ ]:
# 1. Preparamos el modelo definitivo con el mejor K según tu gráfica
mejor_k_n = 10
modelo_k_n= KNeighborsClassifier(n_neighbors=mejor_k_n)

# 2. Entrenamos el modelo con TODO el t_fold (las 120 líneas)
# Usamos X4 e y que ya definiste antes
modelo_k_n.fit(X4, y)

# 3. Preparamos los datos del 'blind fold' (b_fold)
# Extraemos las características (features)
X_blind = b_fold[['sepalo_largo','sepalo_ancho','petalo_largo','petalo_ancho']].values
# Transformamos las etiquetas (species) usando el LabelEncoder que ya tienes
y_blind = le.transform(b_fold['especies'].values)

# 4. El examen final: predecimos sobre datos que el modelo NUNCA vio
predicciones_k_n = modelo_k_n.predict(X_blind)

# 5. Calculamos la precisión real

precision_real_n = accuracy_score(y_blind, predicciones_k_n)

print(f"--- RESULTADO FINAL ---")
print(f"Precisión en el Lote Ciego: {precision_real_n * 100:.2f}%")

# 6. Contamos la cantidad de errores en la predicción
indices_fallo_n = np.where(predicciones_k_n != y_blind)[0]
print('----------------')
print(f"Índices de error con K=10: {indices_fallo_n}")
# total de errores
print(f"Total de errores con K=10: {len(indices_fallo_n)}")

**Observaciones:**

Tras obtener una mejor predicción usando el valor k=10, se procede a hacer una exploración de varios valores en su entorno


### Evaluamos las predicciones para K que pertenezcan a la meseta del grafico y estan en el entorno de k=10.

In [ ]:
# 1. Preparamos los datos del 'blind fold' (b_fold) UNA SOLA VEZ
# Extraemos las características (features)
X_blind = b_fold[['sepalo_largo','sepalo_ancho','petalo_largo','petalo_ancho']].values
# Transformamos las etiquetas (species) usando el LabelEncoder que ya tienes
y_blind = le.transform(b_fold['especies'].values)

print("--- RESULTADOS DE LA EVALUACIÓN ---")

# 2. Iniciamos el ciclo for para K desde 9 hasta 13
# Usamos range(9, 14) porque el límite superior en Python es exclusivo
for mejor_k_n in range(9, 14):

    # 3. Preparamos el modelo con el K actual de la iteración
    modelo_k_n = KNeighborsClassifier(n_neighbors=mejor_k_n)

    # 4. Entrenamos el modelo con TODO el t_fold (las 120 líneas)
    modelo_k_n.fit(X4, y)

    # 5. El examen final: predecimos sobre datos que el modelo NUNCA vio
    predicciones_k_n = modelo_k_n.predict(X_blind)

    # 6. Calculamos la precisión real
    precision_real_n = accuracy_score(y_blind, predicciones_k_n)

    # 7. Mostramos el resultado de cada iteración
    print(f"Para K = {mejor_k_n:2} -> Precisión en el Lote Ciego: {precision_real_n * 100:.2f}%")

    # 8. Contamos la cantidad de errores en la predicción
    indices_fallo_n = np.where(predicciones_k_n != y_blind)[0]
    print(f"Índices de error: {indices_fallo_n}")
    # total de errores
    print(f"Total de errores : {len(indices_fallo_n)}")
    print('----------------')



In [ ]:
    modelo_k_n = KNeighborsClassifier(n_neighbors=10)
    modelo_k_n.fit(X4, y)
    predicciones_k_n = modelo_k_n.predict(X_blind)
    precision_real_n = accuracy_score(y_blind, predicciones_k_n)
    indices_fallo_10 = np.where(predicciones_k_n != y_blind)[0]
    print(f"Índices de error: {indices_fallo_10}")
    # total de errores
    print(f"Total de errores : {len(indices_fallo_n)}")

In [ ]:
# Registros en 12va posicion y 26 de b_fold
b_fold.iloc[[12, 26]]

In [ ]:
# 1. Creamos el pairplot base
g = sns.pairplot(dfIris, hue="especies", diag_kind="kde")

# 2. Filtramos el DataFrame para obtener solo los registros 52 y 106
puntos_destacados = dfIris.loc[[52, 106]]

# 3. Iteramos sobre las variables en X e Y del pairplot para superponer los puntos
for i, y_var in enumerate(g.y_vars):
    for j, x_var in enumerate(g.x_vars):
        # Dibujamos sobre los gráficos de dispersión
        if i != j:
            ax = g.axes[i, j]

            # Dibujamos los puntos seleccionados en color rojo
            ax.scatter(puntos_destacados[x_var],
                       puntos_destacados[y_var],
                       color='red',
                       edgecolor='black', # Borde negro
                       s=100,             # Tamaño del punto
                       zorder=10)         # zorder alto para que queden por encima de los demás puntos


plt.show()

El punto más alto para $K=14$ si bien presenta un máximo para la acuracy con cerca del 98%, son los valores de k=9 y k=10 los que devuelven las  predicciones con menor cantidad de errores.


A primera vista, parece haber una contradicción entre lo que sugiere nuestra validación cruzada donde los K que presentan una acuracy mayor no son los mas performantes al evaluar el lote ciego y si lo son en cambio valores de K mas bajos que se encuentran en una zona de meseta de la gráfica de estudio de la estabilidad.

Nuestra hipotesis es que esto se debe a la combinación del tamaño del lote ciego y cómo cambian las fronteras de decisión en KNN.

Por su parte, el modelo con k=14 devuelve 2 errores y k=9 o 10, devuelve 1 error. Es importante notar que el caso de error para todos los K evaluados, corresponde al caso del registro con indice 52 y 106 del dataset original dfIris y que al graficar estos puntos en el pairplot se pudo observar que se trata de casos limite entre las especies iris-versicolor y iris-virginica.

Entre K=9 y K=13, los modelos entran en una aparente "meseta" y la desviación estándar cae a su mínimo (~0.020). Esto indica que el modelo es robusto y no depende de la suerte de cómo se dividieron los datos.

Consecuentemente con lo esperado los valores para k=9 y k=10 resultan los mejores performantes lo cual es esperable de acuerdo con la teoria para la selección óptima de K como la raiz cuadrada del total de datos utilizados para el entrenamiento.

## Estudio del comportamiento de la precisión y sensibilidad en función de neighbors y cantidad de folds (usando registros de: 4 dimensiones)

In [ ]:
# --- 1. Generación de Datos ---
neighbors_range = list(range(2, 30))
splits_range = [3, 5, 10, 15]

accuracy_matrix = np.zeros((len(splits_range), len(neighbors_range)))
desvest_matrix = np.zeros((len(splits_range), len(neighbors_range)))

# Bucle de entrenamiento
for col_idx, i in enumerate(neighbors_range):
    model_knn = KNeighborsClassifier(n_neighbors=i)
    for row_idx, p in enumerate(splits_range):
        kf = KFold(n_splits=p, shuffle=True, random_state=88)
        # Nota: Asegúrate de tener X4 e y cargados en memoria
        scores = cross_val_score(model_knn, X4, y, cv=kf)

        accuracy_matrix[row_idx, col_idx] = scores.mean()
        desvest_matrix[row_idx, col_idx] = scores.std()

# --- 2. Cálculo del Valor Objetivo y el Óptimo ---
limite_inferior_matrix = accuracy_matrix - (2 * desvest_matrix)

# Encontrar las coordenadas del mejor valor
mejor_valor_objetivo = np.max(limite_inferior_matrix)
indices_mejor = np.unravel_index(np.argmax(limite_inferior_matrix), limite_inferior_matrix.shape)
mejor_fila, mejor_columna = indices_mejor[0], indices_mejor[1]

mejor_split = splits_range[mejor_fila]
mejor_k = neighbors_range[mejor_columna]

X, Y = np.meshgrid(neighbors_range, splits_range)

# --- 3. Visualización Interactiva (3 Paneles) ---

# Crear un lienzo con 3 subgráficos 3D
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{'type': 'surface'}, {'type': 'surface'}, {'type': 'surface'}]],
    subplot_titles=('Accuracy Promedio', 'Desviación Estándar', 'Valor Objetivo (Piso Seguro)')
)

# Panel 1: Accuracy
fig.add_trace(
    go.Surface(z=accuracy_matrix, x=X, y=Y, colorscale='Viridis', colorbar=dict(x=0.27, title='Acc')),
    row=1, col=1
)

# Panel 2: Desviación Estándar
fig.add_trace(
    go.Surface(z=desvest_matrix, x=X, y=Y, colorscale='Plasma', colorbar=dict(x=0.61, title='Std')),
    row=1, col=2
)

# Panel 3: Valor Objetivo
fig.add_trace(
    go.Surface(z=limite_inferior_matrix, x=X, y=Y, colorscale='Tealrose', colorbar=dict(x=1.0, title='Obj')),
    row=1, col=3
)

# Añadir el "Punto Estrella" en el Panel 3 (La mejor configuración)
fig.add_trace(
    go.Scatter3d(
        x=[mejor_k],
        y=[mejor_split],
        z=[mejor_valor_objetivo + 0.005], # Lo subimos un poquito para que no quede hundido en la superficie
        mode='markers+text',
        marker=dict(size=8, color='black', symbol='diamond'),
        text=[f"Óptimo<br>K={mejor_k}, Splits={mejor_split}"],
        textposition="top center",
        name="Mejor Modelo"
    ),
    row=1, col=3
)

# --- 4. Configuración Estética ---
fig.update_layout(
    title_text='Análisis Tridimensional de Hiperparámetros (Exploración Interactiva)',
    height=600,
    width=1600, # Más ancho para acomodar los 3 gráficos
    showlegend=False,
    scene=dict(xaxis_title='K', yaxis_title='Splits', zaxis_title='Accuracy'),
    scene2=dict(xaxis_title='K', yaxis_title='Splits', zaxis_title='Std Dev'),
    scene3=dict(xaxis_title='K', yaxis_title='Splits', zaxis_title='Valor Objetivo')
)

fig.show()

En esta gráfica incluimos la representación del "Piso Seguro" (Límite Inferior). El piso seguro representa el peor escenario de rendimiento que puedes esperar de tu modelo una vez que lo pongas a funcionar en el mundo real.

Estadísticamente, si asumimos una distribución normal de los resultados, 'Promedio'- 2'Desviación Estándar'= límite inferior de un intervalo de confianza del ~95%.
Esto nos permite estar 95% seguros de que el rendimiento de nuestro modelo nunca bajará de ese porcentaje.

Por otro lado, el valor objetivo es la métrica que decidimos que el algoritmo debe intentar maximizar (o minimizar) para declarar a un modelo como ganador.

Al realizar esta evaluación de superficie, introduciendo la penalización por varianza ($Accuracy - 2\sigma$), se detecta que K=20 con 3 Splits como la configuración que se muestra más "resiliente".

## Análisis detallado de K=20 splits=3

In [ ]:
# 1. Calcular la matriz del Límite Inferior (Valor Objetivo)
# La operación se aplica a todos los elementos de las matrices simultáneamente
limite_inferior_matrix = accuracy_matrix - (2 * desvest_matrix)

# 2. Encontrar el valor máximo (el "piso" más alto)
mejor_valor_objetivo = np.max(limite_inferior_matrix)

# 3. Ubicar las coordenadas de ese valor máximo en la matriz
# np.argmax da la posición plana, np.unravel_index la convierte a (fila, columna)
indices_mejor = np.unravel_index(np.argmax(limite_inferior_matrix), limite_inferior_matrix.shape)
mejor_fila = indices_mejor[0]
mejor_columna = indices_mejor[1]

# 4. Traducir esas coordenadas a tus hiperparámetros reales
mejor_split = splits_range[mejor_fila]
mejor_k = neighbors_range[mejor_columna]
accuracy_asociado = accuracy_matrix[mejor_fila, mejor_columna]
desvest_asociada = desvest_matrix[mejor_fila, mejor_columna]

# 5. Imprimir el veredicto final
print("-" * 40)
print("EL MODELO MÁS CONFIABLE PARA PRODUCCIÓN")
print("-" * 40)
print(f"Mejor cantidad de Vecinos (K): {mejor_k}")
print(f"Mejor cantidad de Folds (CV):  {mejor_split}")
print(f"\nMétricas de esa configuración:")
print(f"-> Accuracy Promedio:          {accuracy_asociado:.4f}")
print(f"-> Desviación Estándar:        {desvest_asociada:.4f}")
print(f"-> PISO ESPERADO (95% conf.):  {mejor_valor_objetivo:.4f}")
print("-" * 40)

El modelo con k=20 y split=3 devuelve una precisión promedio esperada igual a la de k=9 y k=10 y split=5 pero con la menor desviación estandar.

## Auditoría de los resultados para k=20 splits=3

In [ ]:
# 1. ENTRENAMIENTO FINAL (Refit)
# Aquí el modelo usa las 120 líneas completas para aprender. Ya no hay splits.
modelo_final_20 = KNeighborsClassifier(n_neighbors=20)
modelo_final_20.fit(X4, y)

# 2. PREPARACIÓN DEL EXAMEN (Blind Fold)
# Usamos las 30 líneas que separaste al puro principio del ejercicio
X_blind = b_fold[['sepalo_largo','sepalo_ancho','petalo_largo','petalo_ancho']].values
y_blind = le.transform(b_fold['especies'].values)

# 3. EL MOMENTO DE LA VERDAD
# El modelo predice sobre datos que NO conoce
predicciones = modelo_final_20.predict(X_blind)

# 4. RESULTADO
score_final = accuracy_score(y_blind, predicciones)

print(f"--- VALIDACIÓN FINAL ---")
print(f"Precisión real del modelo (K=20) ante datos nuevos: {score_final * 100:.2f}%")

### Matriz de confusión

In [ ]:
# 1. Calculamos la matriz con las predicciones (K=20)
cm = confusion_matrix(y_blind, predicciones)

# 2. Creamos la visualización
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu',
            xticklabels=le.classes_,
            yticklabels=le.classes_)

plt.title(f'Matriz de Confusión Final (K=20)\nPrecisión Real: {score_final*100:.2f}%', fontsize=14)
plt.xlabel('Predicción del Modelo', fontsize=12)
plt.ylabel('Clase Real (La Verdad)', fontsize=12)
plt.show()

# 3. Reporte de métricas detallado (Precisión, Recall, F1-score)
print("\nReporte de Clasificación Detallado:")
print(classification_report(y_blind, predicciones, target_names=le.classes_))

Comparación de casos de fallos en modelos k=10, k=14 y k=20

In [ ]:
# Comparación de predicciones falladas con modelo K=20:
indices_fallo_20 = np.where(predicciones != y_blind)[0]
print(f"Índices de flores donde falló el K=10: {indices_fallo_10}")
print(f"Índices de flores donde falló el K=14: {indices_fallo_14}")
print(f"Índices de flores donde falló el K=20: {indices_fallo_20}")

# buscamos el indice de fallos de k=10
modelo_k_n = KNeighborsClassifier(n_neighbors=10)
modelo_k_n.fit(X4, y)
predicciones_k_n = modelo_k_n.predict(X_blind)
precision_real_n = accuracy_score(y_blind, predicciones_k_n)
indices_fallo_10 = np.where(predicciones_k_n != y_blind)[0]

# Cruzamos los datos para ver qué flores son exactamente
#if np.array_equal(indices_fallo_10, indices_fallo_20):
#    print("\nCONCLUSIÓN: Los errores son IDÉNTICOS en ambos modelos.")
#    print("Esto confirma que estas flores están en una zona de solapamiento crítico.")
#else:
#    print("\nCONCLUSIÓN: Los modelos fallan en flores distintas.")

# 4. Ver qué flores son (especie real vs predicha)
print("\n--- Detalle de los fallos (K=10) ---")
for idx in indices_fallo_10:
    real = le.inverse_transform([y_blind[idx]])[0]
    predicho = le.inverse_transform([predicciones[idx]])[0]
    print(f"Registro {idx}: Era '{real}' pero el modelo dijo '{predicho}'")

print("\n--- Detalle de los fallos (K=14) ---")
for idx in indices_fallo_14:
    real = le.inverse_transform([y_blind[idx]])[0]
    predicho = le.inverse_transform([predicciones[idx]])[0]
    print(f"Registro {idx}: Era '{real}' pero el modelo dijo '{predicho}'")



print("\n--- Detalle de los fallos (K=20) ---")
for idx in indices_fallo_20:
    real = le.inverse_transform([y_blind[idx]])[0]
    predicho = le.inverse_transform([predicciones[idx]])[0]
    print(f"Registro {idx}: Era '{real}' pero el modelo dijo '{predicho}'")

# RESULTADO:

Los modelos con mayor acuracy (k=14) y con mejor límite inferior de confianza (k=20) fallaron en detectar uno de los casos límites (el caso con index 52 del dataset original que hemos graficado previamente con pairplot y destacado con puntos rojos) mientras que un modelo con peor límite inferior de confianza (k=9, k=10 con splits=5) lo detecta sin problema.


# CONCLUSIÓN:

La elección de modelos cuyos hiperparámetros se ubican en una meseta o zona de estabilidad obtuvieron mejores predicciones al ser auditados con los datasets ciegos, puntualmente en la detección de casos límites respecto a otros modelos que teóricamente poseen una mejor acuracy o mejor límite inferior de confianza. Por ende, podriamos concluir que modelos mas estables son en definitiva al contrastarlos con la "realidad" son mejores opciones como modelos de producción.


# Discusión

El dataset Iris, tiene 150 registros y al ser particionado en 80/20, nos deja con 120 registros para realizar el entrenamiento.
Es posible que la diferencia entre los resultados teóricos y los prácticos obtenidos, respondan a limitado tamaño del universo de datos y esto no sea trasladable a escenarios con mayor cantidad de datos disponibles.